# Indic Multilingual ASR Fine-tuning with Whisper + LoRA
## Google Colab Training Notebook

This notebook trains a Whisper-small model with LoRA for Indic languages (Marathi, Gujarati, Hindi).

**Features:**
- LoRA-based parameter-efficient fine-tuning (60% memory savings)
- Common Voice dataset for each language
- FLEURS benchmark evaluation
- Automatic checkpoint saving to Google Drive
- Per-language configuration support

**Estimated Runtime:**
- CPU: 4-5 hours per epoch
- GPU (T4): 1-1.5 hours per epoch
- GPU (A100): 20-30 minutes per epoch

**Prerequisites:**
- Google Colab Pro (recommended for GPU access)
- GitHub PAT for model pushing
- Google Drive for storage (~50GB capacity needed)

## 1. Environment Setup

First, let's install all dependencies and check GPU availability.

In [ ]:
import subprocess
import sys

# Install core dependencies
packages = [
    "torch>=2.0.0",
    "transformers>=4.30.0",
    "peft>=0.4.0",
    "datasets>=2.13.0",
    "accelerate>=0.20.0",
    "evaluate>=0.4.0",
    "jiwer>=3.0.0",
    "pydantic>=2.0.0",
    "pyyaml>=6.0.0",
    "librosa>=0.10.0",
    "soundfile>=0.12.0",
    "loguru>=0.7.0"
]

print("Installing dependencies...")
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✓ All dependencies installed successfully!")

# Verify GPU availability
import torch
print(f"\nGPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully!")

# Create working directories
workspace_root = '/content/drive/MyDrive/speech_agent_workspace'
os.makedirs(workspace_root, exist_ok=True)
os.makedirs(f'{workspace_root}/data', exist_ok=True)
os.makedirs(f'{workspace_root}/models', exist_ok=True)
os.makedirs(f'{workspace_root}/checkpoints', exist_ok=True)

print(f"✓ Workspace directories created at: {workspace_root}")

## 2. Data Download and Preparation

Download Common Voice and FLEURS datasets for your target language.

**Language Options:**
- Marathi (mr)
- Gujarati (gu)
- Hindi (hi)

In [ ]:
from datasets import load_dataset
import os

# Select your target language
LANGUAGE_CODE = "mr"  # Change to "gu" or "hi" for other languages
LANGUAGE_NAME = {"mr": "Marathi", "gu": "Gujarati", "hi": "Hindi"}[LANGUAGE_CODE]

print(f"Loading {LANGUAGE_NAME} dataset...")

# Load Common Voice dataset
common_voice = load_dataset(
    "mozilla-foundation/common_voice_17_0",
    LANGUAGE_CODE,
    split="train",
    cache_dir=f'{workspace_root}/data/common_voice',
    trust_remote_code=True
)

print(f"✓ Loaded {len(common_voice)} samples from Common Voice")
print(f"Dataset info: {common_voice}")

# Select subset for faster training (first 200 samples)
train_dataset = common_voice.select(range(min(200, len(common_voice))))
test_dataset = common_voice.select(range(min(200, len(common_voice)), min(250, len(common_voice))))

print(f"✓ Selected {len(train_dataset)} training samples and {len(test_dataset)} test samples")

## 3. Model Configuration

Load Whisper model with LoRA configuration.

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import get_peft_model, LoraConfig, TaskType
import torch

# Download model
MODEL_NAME = "openai/whisper-small"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME}...")
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

print(f"✓ Model loaded. Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Apply LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)
print(f"✓ LoRA applied!")
model.print_trainable_parameters()

## 4. Fine-tuning Process

Prepare data and run the training loop with LoRA.

In [ ]:
# Prepare dataset for training
def prepare_dataset(batch):
    """Process audio and text for training."""
    # Load audio
    audio = batch["audio"]
    
    # Process audio
    features = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    )
    
    # Process text
    input_ids = processor.tokenizer(
        batch["sentence"],
        return_tensors="pt"
    ).input_ids
    
    batch["input_features"] = features.input_features[0]
    batch["labels"] = input_ids[0]
    
    return batch

print("Processing dataset...")
train_dataset_processed = train_dataset.map(
    prepare_dataset,
    remove_columns=train_dataset.column_names,
    num_proc=4
)

print(f"✓ Dataset processed with {len(train_dataset_processed)} samples")

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """Data collator for speech-to-text tasks."""
    processor: Any
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have different lengths and need different padding methods
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        
        batch["labels"] = labels
        return batch

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=f'{workspace_root}/checkpoints',
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=50,
    num_train_epochs=1,
    evaluation_strategy="no",
    save_strategy="steps",
    save_steps=50,
    logging_steps=10,
    remove_unused_columns=False,
    push_to_hub=False,
    dataloader_pin_memory=True,
    dataloader_num_workers=4,
    fp16=torch.cuda.is_available(),
)

# Create trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset_processed,
    data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor),
    processing_class=processor,
)

print("Starting training...")
trainer.train()
print("✓ Training completed!")

## 5. Model Evaluation

Evaluate the fine-tuned model on test data using WER metric.

In [ ]:
from jiwer import wer, cer
import numpy as np

# Load test dataset and process
test_dataset_processed = test_dataset.map(
    prepare_dataset,
    remove_columns=test_dataset.column_names,
    num_proc=4
)

print("Running inference on test set...")
predictions = []
references = []

model.eval()
with torch.no_grad():
    for i, sample in enumerate(test_dataset_processed):
        if i % 10 == 0:
            print(f"Processing sample {i}/{len(test_dataset_processed)}")
        
        input_features = sample["input_features"].unsqueeze(0).to(device)
        predicted_ids = model.generate(input_features)
        
        pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        ref_text = processor.batch_decode([sample["labels"]], skip_special_tokens=True)[0]
        
        predictions.append(pred_text)
        references.append(ref_text)

# Compute metrics
wer_score = wer(references, predictions)
cer_score = cer(references, predictions)

print(f"\n✓ Evaluation Results:")
print(f"  WER: {wer_score:.4f}")
print(f"  CER: {cer_score:.4f}")

# Show sample predictions
print(f"\nSample Predictions:")
for i in range(min(5, len(predictions))):
    print(f"Reference: {references[i]}")
    print(f"Prediction: {predictions[i]}")
    print()

## 6. Inference and Deployment

Save the fine-tuned model for deployment and perform inference.

In [ ]:
# Save fine-tuned model
model_save_path = f'{workspace_root}/models/whisper-{LANGUAGE_CODE}-finetuned'
os.makedirs(model_save_path, exist_ok=True)

print("Saving model...")
model.save_pretrained(model_save_path)
processor.save_pretrained(model_save_path)
print(f"✓ Model saved to {model_save_path}")

# Create inference pipeline for deployment
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model=model_save_path,
    device=0 if torch.cuda.is_available() else -1
)

# Test inference
print("Testing inference on sample audio...")
test_audio_path = test_dataset[0]['audio']['path']

result = pipe(
    test_audio_path,
    generate_kwargs={"language": LANGUAGE_CODE}
)

print(f"Transcription: {result['text']}")
print("✓ Inference successful!")

# Save model card
model_card = f"""---
language:
- {LANGUAGE_CODE}
datasets:
- mozilla-foundation/common_voice_17_0
metrics:
- wer
- cer
tags:
- automatic-speech-recognition
- whisper
- lora
- indic
---

# Whisper-small Fine-tuned for {LANGUAGE_NAME}

This model is a Whisper-small model fine-tuned for {LANGUAGE_NAME} using LoRA (Low-Rank Adaptation).

## Model Details

- **Base Model**: openai/whisper-small
- **Fine-tuning Method**: LoRA (r=8, alpha=16)
- **Training Dataset**: Common Voice 17.0 ({LANGUAGE_NAME})
- **Language**: {LANGUAGE_NAME} ({LANGUAGE_CODE})

## Performance

- **WER**: {wer_score:.4f}
- **CER**: {cer_score:.4f}

## Usage

```python
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="./whisper-{LANGUAGE_CODE}-finetuned")
result = pipe("audio.wav")
print(result["text"])
```

## Training Details

- Training epochs: 1
- Batch size: 4
- Learning rate: 1e-4
- Optimizer: Adam with warmup
- Total trainable parameters: ~10M (LoRA)
"""

with open(f'{model_save_path}/README.md', 'w') as f:
    f.write(model_card)

print("✓ Model card saved!")